## 02 ARPDAU Deep-Dive: "ARPDAU is falling. Fix it."

Overall ARPDAU declined by approximately 16% from the first 30 days to the last 30 days, while ARPDAU within every `country_tier` remained flat or improved.

**Goal:** Determine whether the decline reflects weaker monetization within country tiers or a change in the composition of the active-player population.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.db import get_connection

conn = get_connection()

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("Database connected successfully.")

Database connected successfully.


In [2]:
q = """
WITH active AS (
    SELECT pd.device_id, pd.activity_date AS act_date
    FROM player_day pd

    UNION

    SELECT
        pu.device_id,
        CAST(strftime('%Y%m%d', pu.event_ts) AS INTEGER) AS act_date
    FROM purchase pu
),
period AS (
    SELECT
        device_id,
        act_date,
        CASE
            WHEN act_date BETWEEN 20260101 AND 20260130 THEN 'first_30'
            WHEN act_date BETWEEN 20260401 AND 20260430 THEN 'last_30'
        END AS period
    FROM active
    WHERE act_date BETWEEN 20260101 AND 20260130
       OR act_date BETWEEN 20260401 AND 20260430
),
dau AS (
    SELECT
        period,
        COUNT(*) AS player_days
    FROM period
    GROUP BY period
),
rev AS (
    SELECT
        CASE
            WHEN CAST(strftime('%Y%m%d', event_ts) AS INTEGER)
                 BETWEEN 20260101 AND 20260130 THEN 'first_30'
            WHEN CAST(strftime('%Y%m%d', event_ts) AS INTEGER)
                 BETWEEN 20260401 AND 20260430 THEN 'last_30'
        END AS period,
        SUM(usd_amount) AS revenue
    FROM purchase
    WHERE CAST(strftime('%Y%m%d', event_ts) AS INTEGER)
              BETWEEN 20260101 AND 20260130
       OR CAST(strftime('%Y%m%d', event_ts) AS INTEGER)
              BETWEEN 20260401 AND 20260430
    GROUP BY period
)
SELECT
    d.period,
    d.player_days,
    r.revenue,
    r.revenue * 1.0 / d.player_days AS arpdau
FROM dau d
JOIN rev r
    ON d.period = r.period
"""

df_overall = pd.read_sql_query(q, conn)

display(df_overall)

first = df_overall.loc[
    df_overall["period"] == "first_30", "arpdau"
].iloc[0]

last = df_overall.loc[
    df_overall["period"] == "last_30", "arpdau"
].iloc[0]

overall_change_pct = (last / first - 1) * 100

print(f"Overall ARPDAU: {first:.4f} -> {last:.4f}")
print(f"Overall ARPDAU change: {overall_change_pct:.1f}%")

,period,player_days,revenue,arpdau
0,first_30,82716,28926.91,0.349714
1,last_30,283805,82925.47,0.292192


Overall ARPDAU: 0.3497 -> 0.2922
Overall ARPDAU change: -16.4%


In [3]:
q_tier = """
WITH active AS (
    SELECT pd.device_id, pd.activity_date AS act_date
    FROM player_day pd

    UNION

    SELECT
        pu.device_id,
        CAST(strftime('%Y%m%d', pu.event_ts) AS INTEGER) AS act_date
    FROM purchase pu
),
period AS (
    SELECT
        a.device_id,
        a.act_date,
        pp.country_tier,
        CASE
            WHEN a.act_date BETWEEN 20260101 AND 20260130 THEN 'first_30'
            WHEN a.act_date BETWEEN 20260401 AND 20260430 THEN 'last_30'
        END AS period
    FROM active a
    JOIN player_profile pp
        ON a.device_id = pp.device_id
    WHERE a.act_date BETWEEN 20260101 AND 20260130
       OR a.act_date BETWEEN 20260401 AND 20260430
),
dau AS (
    SELECT
        period,
        country_tier,
        COUNT(*) AS player_days
    FROM period
    GROUP BY period, country_tier
),
rev AS (
    SELECT
        CASE
            WHEN CAST(strftime('%Y%m%d', pu.event_ts) AS INTEGER)
                 BETWEEN 20260101 AND 20260130 THEN 'first_30'
            WHEN CAST(strftime('%Y%m%d', pu.event_ts) AS INTEGER)
                 BETWEEN 20260401 AND 20260430 THEN 'last_30'
        END AS period,
        pp.country_tier,
        SUM(pu.usd_amount) AS revenue
    FROM purchase pu
    JOIN player_profile pp
        ON pu.device_id = pp.device_id
    WHERE CAST(strftime('%Y%m%d', pu.event_ts) AS INTEGER)
              BETWEEN 20260101 AND 20260130
       OR CAST(strftime('%Y%m%d', pu.event_ts) AS INTEGER)
              BETWEEN 20260401 AND 20260430
    GROUP BY period, pp.country_tier
)
SELECT
    d.country_tier,
    d.period,
    d.player_days,
    COALESCE(r.revenue, 0) AS revenue,
    COALESCE(r.revenue, 0) * 1.0 / d.player_days AS arpdau
FROM dau d
LEFT JOIN rev r
    ON d.period = r.period
   AND d.country_tier = r.country_tier
ORDER BY d.country_tier, d.period
"""

df_tier = pd.read_sql_query(q_tier, conn)

piv = df_tier.pivot(
    index="country_tier",
    columns="period",
    values="arpdau"
)

piv["pct_change"] = (
    (piv["last_30"] / piv["first_30"]) - 1
) * 100

display(piv.round(4))

period,first_30,last_30,pct_change
country_tier,,,
1,0.5156,0.5419,5.1202
2,0.3611,0.4002,10.8224
3,0.2311,0.2625,13.6016
4,0.1545,0.1554,0.6125


## ARPDAU Decomposition

To explain the contradiction between falling overall ARPDAU and stable/improving tier-level ARPDAU, the total change is separated into:

- **Within-tier effect:** change caused by ARPDAU changing within country tiers.
- **Mix effect:** change caused by the distribution of player-days shifting across country tiers.

In [4]:
shares = df_tier.pivot(
    index="country_tier",
    columns="period",
    values="player_days"
)

shares["first_30_share"] = (
    shares["first_30"] / shares["first_30"].sum()
)

shares["last_30_share"] = (
    shares["last_30"] / shares["last_30"].sum()
)

print("Player-day mix by country tier:")
display(
    shares[["first_30_share", "last_30_share"]]
    .mul(100)
    .round(2)
)

merged = piv.join(
    shares[["first_30_share", "last_30_share"]]
)

actual_first = (
    merged["first_30_share"] *
    merged["first_30"]
).sum()

actual_last = (
    merged["last_30_share"] *
    merged["last_30"]
).sum()

# Counterfactual:
# last-period tier ARPDAU with first-period player mix
counterfactual = (
    merged["first_30_share"] *
    merged["last_30"]
).sum()

within_tier_effect = counterfactual - actual_first
mix_effect = actual_last - counterfactual
total_change = actual_last - actual_first

print(f"Actual blended ARPDAU: {actual_first:.4f} -> {actual_last:.4f}")
print(f"Counterfactual ARPDAU: {counterfactual:.4f}")

print(f"\nWithin-tier effect: {within_tier_effect:+.4f}")
print(f"Mix effect:         {mix_effect:+.4f}")
print(f"Total change:       {total_change:+.4f}")

print(
    f"\nMix effect relative to net ARPDAU decline: "
    f"{mix_effect / total_change * 100:.1f}% "
    "(above 100% because within-tier improvement partially offset the negative mix effect)"
)

Player-day mix by country tier:


period,first_30_share,last_30_share
country_tier,,
1,34.53,20.54
2,26.07,15.57
3,21.78,17.99
4,17.62,45.90


Actual blended ARPDAU: 0.3497 -> 0.2922
Counterfactual ARPDAU: 0.3760

Within-tier effect: +0.0263
Mix effect:         -0.0838
Total change:       -0.0575

Mix effect relative to net ARPDAU decline: 145.7% (above 100% because within-tier improvement partially offset the negative mix effect)


## Investigating the Mix Shift

The decomposition indicates that player composition is important. Next, examine whether changes in new-player acquisition are associated with the shift in country-tier mix.

In [5]:
q_installs = """
SELECT
    CASE
        WHEN install_date BETWEEN '2026-01-01' AND '2026-01-30'
            THEN 'first_30_installs'
        WHEN install_date BETWEEN '2026-04-01' AND '2026-04-30'
            THEN 'last_30_installs'
    END AS period,
    country_tier,
    COUNT(*) AS n_installs
FROM player_profile
WHERE install_date BETWEEN '2026-01-01' AND '2026-01-30'
   OR install_date BETWEEN '2026-04-01' AND '2026-04-30'
GROUP BY period, country_tier
ORDER BY period, country_tier
"""

df_installs = pd.read_sql_query(q_installs, conn)

piv_installs = df_installs.pivot(
    index="country_tier",
    columns="period",
    values="n_installs"
)

piv_installs["first_30_pct"] = (
    piv_installs["first_30_installs"] /
    piv_installs["first_30_installs"].sum() * 100
)

piv_installs["last_30_pct"] = (
    piv_installs["last_30_installs"] /
    piv_installs["last_30_installs"].sum() * 100
)

print("New-install country-tier mix:")
display(
    piv_installs[
        ["first_30_pct", "last_30_pct"]
    ].round(1)
)

New-install country-tier mix:


period,first_30_pct,last_30_pct
country_tier,,
1,33.8,12.1
2,25.8,9.3
3,22.2,15.9
4,18.1,62.7


In [6]:
q_channel = """
SELECT
    CASE
        WHEN install_date BETWEEN '2026-01-01' AND '2026-01-30'
            THEN 'first_30_installs'
        WHEN install_date BETWEEN '2026-04-01' AND '2026-04-30'
            THEN 'last_30_installs'
    END AS period,
    acquisition_channel,
    COUNT(*) AS n_installs
FROM player_profile
WHERE install_date BETWEEN '2026-01-01' AND '2026-01-30'
   OR install_date BETWEEN '2026-04-01' AND '2026-04-30'
GROUP BY period, acquisition_channel
ORDER BY period, acquisition_channel
"""

df_channel = pd.read_sql_query(q_channel, conn)

piv_channel = df_channel.pivot(
    index="acquisition_channel",
    columns="period",
    values="n_installs"
)

piv_channel["first_30_pct"] = (
    piv_channel["first_30_installs"] /
    piv_channel["first_30_installs"].sum() * 100
)

piv_channel["last_30_pct"] = (
    piv_channel["last_30_installs"] /
    piv_channel["last_30_installs"].sum() * 100
)

print("Acquisition-channel mix:")
display(
    piv_channel[
        ["first_30_pct", "last_30_pct"]
    ].round(1)
)

Acquisition-channel mix:


period,first_30_pct,last_30_pct
acquisition_channel,,
organic,45.8,16.4
paid_social,39.0,14.4
paid_video,15.2,69.2


In [7]:
q_cross = """
SELECT
    acquisition_channel,
    country_tier,
    COUNT(*) AS n_installs
FROM player_profile
WHERE install_date BETWEEN '2026-04-01' AND '2026-04-30'
GROUP BY acquisition_channel, country_tier
ORDER BY acquisition_channel, country_tier
"""

df_cross = pd.read_sql_query(q_cross, conn)

piv_cross = df_cross.pivot(
    index="acquisition_channel",
    columns="country_tier",
    values="n_installs"
).fillna(0)

piv_cross_pct = (
    piv_cross.div(piv_cross.sum(axis=1), axis=0) * 100
)

print("April installs: country-tier % within each channel")
display(piv_cross_pct.round(1))

April installs: country-tier % within each channel


country_tier,1,2,3,4
acquisition_channel,,,,
organic,32.2,26.1,22.7,19.0
paid_social,34.2,26.0,21.6,18.2
paid_video,2.7,1.9,13.1,82.3


In [8]:
df_overall.to_csv(
    "../outputs/tables/02_arpdau_overall.csv",
    index=False
)

df_tier.to_csv(
    "../outputs/tables/02_arpdau_by_tier.csv",
    index=False
)

shares.reset_index().to_csv(
    "../outputs/tables/02_country_tier_mix.csv",
    index=False
)

df_installs.to_csv(
    "../outputs/tables/02_install_tier_mix.csv",
    index=False
)

df_channel.to_csv(
    "../outputs/tables/02_acquisition_channel_mix.csv",
    index=False
)

piv_cross_pct.reset_index().to_csv(
    "../outputs/tables/02_channel_tier_mix.csv",
    index=False
)

print("All Part 2 outputs saved successfully.")

All Part 2 outputs saved successfully.


## Findings and Recommendation

- Overall ARPDAU declined by approximately **16.4%** from the first 30 days to the last 30 days.
- ARPDAU within every country tier remained flat or improved.
- The decomposition shows that the blended decline is primarily associated with a **shift in the country-tier mix**, while the within-tier effect is positive.
- `paid_video` grew substantially as a share of new installs and its April installs were heavily concentrated in Tier 4, making it a plausible contributor to the observed mix shift.

**Recommendation:** Do not reduce acquisition based on falling blended ARPDAU alone. Monitor ARPDAU by country tier and acquisition channel, and evaluate acquisition decisions using player value/LTV relative to acquisition cost.

**Limitation:** The dataset does not contain marketing spend or cost-per-install data, so it cannot establish whether `paid_video` or Tier 4 acquisition is profitable or unprofitable.